# FASE 1: Carga y Entendimiento del Dataset BVG

Este notebook se enfoca unicamente en la Fase 1 del pipeline de tesis:
- Carga de datos desde URL https://www.bolsadevaloresguayaquil.com/boletines/historicos/BVG_Acciones.xlsx
- Inspeccion general del dataset
- Filtrado de empresas objetivo
- Validaciones basicas de estructura temporal

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 250)
sns.set_theme(style='whitegrid')

In [2]:
# Abrir archivo con pandas
df = pd.read_csv('data/raw/BVG_Acciones.csv', sep=';')

print(f'Filas cargadas: {len(df):,}')
print(f'Columnas cargadas: {len(df.columns)}')

Filas cargadas: 28,592
Columnas cargadas: 11


In [3]:
# Inspeccion general del dataset
print('Shape del dataset:', df.shape)
print('\nColumnas disponibles:')
display(df.columns.tolist())
print('\nTipos de datos:')
display(df.dtypes)

print('\nPrimeras filas:')
display(df.head())
print('\nNulos por columna:')
display(df.isna().sum())

Shape del dataset: (28592, 11)

Columnas disponibles:


['FECHA NEGOCIACIÓN',
 'TÍTULO',
 'EMISOR',
 'NÚMERO DE ACCIONES',
 'V. NOM. UNITARIO',
 'PRECIO',
 'VALOR NOMINAL',
 'VALOR EFECTO',
 'CASA COMPRADORA',
 'CASA VENDEDORA',
 'BOLSA']


Tipos de datos:


FECHA NEGOCIACIÓN         str
TÍTULO                    str
EMISOR                    str
NÚMERO DE ACCIONES        str
V. NOM. UNITARIO          str
PRECIO                float64
VALOR NOMINAL             str
VALOR EFECTO              str
CASA COMPRADORA           str
CASA VENDEDORA            str
BOLSA                     str
dtype: object


Primeras filas:


,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
0,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,1.000.00,1.00,2.45,1.000.00,2.450.00,MERCAPITAL,ORION,BVG
1,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,821.00,1.00,2.44,821.00,2.003.24,PICAVAL,PICAVAL,BVG
2,02/01/2019,ACCIONES,SURPAPELCORP S.A.,90.00,1.00,4.25,90.00,382.50,ACCIONES Y VALORES,ACCIONES Y VALORES,BVG
3,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,2.027.00,1.00,2.44,2.027.00,4.945.88,SANTA FE,PICAVAL,BVG
4,02/01/2019,ACCIONES,CORPORACION FAVORITA C.A.,189.00,1.00,2.44,189.00,461.16,SILVERCROSS,PICAVAL,BVG



Nulos por columna:


FECHA NEGOCIACIÓN     0
TÍTULO                0
EMISOR                0
NÚMERO DE ACCIONES    0
V. NOM. UNITARIO      0
PRECIO                0
VALOR NOMINAL         0
VALOR EFECTO          0
CASA COMPRADORA       0
CASA VENDEDORA        0
BOLSA                 0
dtype: int64

In [4]:
# Variables necesarias
emisor_col = 'EMISOR'
fecha_col = 'FECHA NEGOCIACIÓN'
precio_col = 'PRECIO'

In [5]:
# Filtrado por empresas objetivo
empresas_objetivo = ['CORPORACION FAVORITA C.A.', 'BANCO GUAYAQUIL S.A.']
df_obj = df[df[emisor_col].isin(empresas_objetivo)].copy()

print('Shape del dataset filtrado:', df_obj.shape)
print('Registros por empresa:')
display(df_obj[emisor_col].value_counts())

Shape del dataset filtrado: (18188, 11)
Registros por empresa:


EMISOR
CORPORACION FAVORITA C.A.    14706
BANCO GUAYAQUIL S.A.          3482
Name: count, dtype: int64

In [6]:
# Conversion formato de fechas str a datetime (dd/mm/yyyy)
df_obj[fecha_col] = pd.to_datetime(
  df_obj[fecha_col], 
  format='%d/%m/%Y',
  errors='coerce')

fechas_invalidas = df_obj[fecha_col].isna().sum()
print(f'Fechas no convertibles (NaT): {fechas_invalidas}')
df_obj = df_obj.dropna(subset=[fecha_col]).copy()

df_obj[precio_col] = pd.to_numeric(df_obj[precio_col], errors='coerce')
precios_invalidos = df_obj[precio_col].isna().sum()
print(f'Precios no convertibles (NaN): {precios_invalidos}')
df_obj = df_obj.dropna(subset=['PRECIO']).copy()


print(f'Registros luego de conversión: {len(df_obj):,}')
display(df_obj.dtypes)

Fechas no convertibles (NaT): 0
Precios no convertibles (NaN): 0
Registros luego de conversión: 18,188


FECHA NEGOCIACIÓN     datetime64[us]
TÍTULO                           str
EMISOR                           str
NÚMERO DE ACCIONES               str
V. NOM. UNITARIO                 str
PRECIO                       float64
VALOR NOMINAL                    str
VALOR EFECTO                     str
CASA COMPRADORA                  str
CASA VENDEDORA                   str
BOLSA                            str
dtype: object

In [7]:
# Ordenamiento temporal
df_obj = df_obj.sort_values([emisor_col, fecha_col]).reset_index(drop=True)
display(df_obj.head())
display(df_obj.tail())

,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
0,2019-01-02,ACCIONES,BANCO GUAYAQUIL S.A.,2.000.00,1.00,0.96,2.000.00,1.920.00,PLUSBURSÁTIL,SANTA FE,BVG
1,2019-01-03,ACCIONES,BANCO GUAYAQUIL S.A.,11.069.00,1.00,0.96,11.069.00,10.626.24,SILVERCROSS,SILVERCROSS,BVG
2,2019-01-03,ACCIONES,BANCO GUAYAQUIL S.A.,20.035.00,1.00,0.96,20.035.00,19.233.60,SILVERCROSS,SILVERCROSS,BVG
3,2019-01-14,ACCIONES,BANCO GUAYAQUIL S.A.,1.978.00,1.00,0.96,1.978.00,1.898.88,SANTA FE,SANTA FE,BVG
4,2019-01-14,ACCIONES,BANCO GUAYAQUIL S.A.,13.022.00,1.00,0.95,13.022.00,12.370.90,SANTA FE,ORION,BVG


,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
18183,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,1.191.00,1,2.0,1.191.00,2.382.00,ECUABURSÁTIL,MERCAPITAL,BVQ
18184,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,4.000.00,1,2.0,4.000.00,8.000.00,ECUABURSÁTIL,ECUABURSÁTIL,BVQ
18185,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,2.541.00,1,2.0,2.541.00,5.082.00,ECUABURSÁTIL,METROVALORES,BVQ
18186,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,316.00,1,2.0,316.00,632.00,ECUABURSÁTIL,SANTA FE,BVG
18187,2026-03-25,ACCIONES,CORPORACION FAVORITA C.A.,3.201.00,1,2.0,3.201.00,6.402.00,ECUABURSÁTIL,SANTA FE,BVG


In [8]:
# Analisis de rango de fechas y numero total de registros por empresa
resumen_empresas = (
    df_obj.groupby(emisor_col)
    .agg(
        total_registros=(emisor_col, 'size'),
        fecha_min=(fecha_col, 'min'),
        fecha_max=(fecha_col, 'max')
    )
    .sort_values('total_registros', ascending=False)
)

display(resumen_empresas)

,total_registros,fecha_min,fecha_max
EMISOR,,,
CORPORACION FAVORITA C.A.,14706,2019-01-02,2026-03-25
BANCO GUAYAQUIL S.A.,3482,2019-01-02,2026-03-25


In [9]:
# Analisis de completitud e irregularidad temporal (sin imputar ni reindexar)
analisis = []

for empresa, g in df_obj.groupby(emisor_col):
    g = g.sort_values(fecha_col)
    deltas = g[fecha_col].diff().dropna().dt.days

    info = {
        'empresa': empresa,
        'registros': len(g),
        'fecha_min': g[fecha_col].min(),
        'fecha_max': g[fecha_col].max(),
        'mediana_gap_dias': deltas.median() if not deltas.empty else np.nan,
        'p90_gap_dias': deltas.quantile(0.90) if not deltas.empty else np.nan,
        'max_gap_dias': deltas.max() if not deltas.empty else np.nan
    }
    analisis.append(info)

analisis_df = pd.DataFrame(analisis)
display(analisis_df)

,empresa,registros,fecha_min,fecha_max,mediana_gap_dias,p90_gap_dias,max_gap_dias
0,BANCO GUAYAQUIL S.A.,3482,2019-01-02,2026-03-25,0.0,3.0,24
1,CORPORACION FAVORITA C.A.,14706,2019-01-02,2026-03-25,0.0,1.0,10


In [10]:
# Guardar dataset limpio para fase 2
df_obj.to_csv('data/processed/BVG_Acciones_limpio.csv', index=False)

## Conclusión de resultados

### 1. Carga

- El dataset fue cargado correctamente desde data con shape global de `(28592, 11)`.
- El subset de empresas objetivo contiene `18,188` registros.

### 2. Diferencias entre empresas

- Existe desbalance fuerte entre emisores objetivo:
  - `CORPORACION FAVORITA C.A.`: `14706` registros
  - `BANCO GUAYAQUIL S.A.`: `3482` registros
- La irregularidad temporal no es equivalente entre emisores:
  - Banco Guayaquil presenta mayor dispersion de gaps (`p90 = 3 dias`, `max = 24 dias`).
  - Corporacion Favorita muestra una serie mas densa (`p90 = 1 dia`, `max = 10 dias`).

### 3. Interpretacion real de metricas

- Cobertura temporal: ambas empresas abarcan exactamente el mismo rango (`2019-01-02` a `2026-03-25`).
- Completitud relativa: la cantidad de observaciones sugiere mejor estabilidad estadistica para Corporacion Favorita que para Banco Guayaquil.
- La mediana de gap en `0` dias para ambas empresas indica multiples registros por fecha en el dataset transaccional (no necesariamente frecuencia diaria uniforme).
- Conclusión de calidad inicial: evidencia clara de heterogeneidad de liquidez/actividad entre emisores.
